# Python. Работа с JSON и XML

## Мотивация

Данные редко приходят сразу в удобном массиве: API отвечает JSON, корпоративные системы отдают XML, а нужная информация иногда остаётся только в HTML. Цель семинара — получить исходные данные, проверить их структуру и сохранить результат так, чтобы обработку можно было повторить и проверить.

## 0. Подготовка

`json`, `xml.etree.ElementTree`, `os` и `tempfile` входят в Python. Для HTTP, HTML, OpenAI-compatible API и учебного сервера нужны внешние пакеты:

```bash
uv add requests beautifulsoup4 openai fastapi uvicorn
```

Имя пакета и имя импорта иногда различаются: пакет `beautifulsoup4` импортируется как `bs4`. Ниже один раз создаём рабочий каталог и маленькую функцию показа текстового файла. Это служебный код: он не относится к JSON, XML или HTTP и дальше не будет заслонять примеры по теме. `work_path("data.json")` возвращает имя временного файла.

In [ ]:
import os
import tempfile

TEMP_DIRECTORY = tempfile.TemporaryDirectory()
WORK = TEMP_DIRECTORY.name

def work_path(name):
    """Получить путь к временному демонстрационному файлу."""
    return os.path.join(WORK, name)

def show_text(path):
    """Показать небольшой текстовый файл."""
    with open(path, encoding="utf-8") as file:
        print(file.read())

## 1. JSON: зачем он нужен и как его читать

Обычный TXT хранит только символы. Если записать `Анна 86 true`, программа не знает, где имя, где балл, является ли `true` текстом и как представить несколько студентов. Можно придумать свои разделители и правила, но тогда каждый автор создаст собственный формат.

JSON — тоже текст, поэтому его можно открыть в редакторе и передать по сети, но структура и типы записываются по общим правилам. Поля имеют имена, объекты и списки могут быть вложенными, а число, строка, логическое значение и отсутствие значения различаются. Благодаря этому один файл одинаково понимают Python, JavaScript и многие другие языки. Цена удобства — строгий синтаксис и несколько лишних символов.

Соответствие JSON и Python:

| JSON | Python |
|---|---|
| объект `{...}` | словарь `dict` |
| массив `[...]` | список `list` |
| строка, число | `str`, `int` или `float` |
| `true`, `false` | `True`, `False` |
| `null` | `None` |

Названия функций отличаются буквой `s`, которая означает **string**:

- `json.loads(text)` — разобрать JSON-строку;
- `json.dumps(data)` — получить JSON-строку;
- `json.load(file)` — прочитать JSON из уже *открытого* файла;
- `json.dump(data, file)` — записать JSON в уже *открытый* файл.

`ensure_ascii=False` оставляет кириллицу читаемой, `indent=2`(а также 3,4...) добавляет отступы для читабельности. Нативный формат JSON не допускает комментарии, одинарные кавычки и запятую после последнего элемента (но есть кастомные ридеры, которые это допускают).

In [ ]:
import json

students = {
    "course": "Linux и Python",
    "active": True,
    "students": [
        {"name": "Анна", "score": 86},
        {"name": "Илья", "score": 73},
    ],
}

path = work_path("students.json")
with open(path, "w", encoding="utf-8") as file:
    json.dump(students, file, ensure_ascii=False, indent=2)

with open(path, encoding="utf-8") as file:
    loaded = json.load(file)

show_text(path)
print(type(loaded), type(loaded["students"]))
print(loaded["students"][0]["score"])

#### ❓ **Вопрос**

Что возвращают `json.load(file)` и `json.loads(text)`? Почему работает `loaded["students"][0]["score"]`?

<details><summary>Ответ</summary>

Обе функции создают обычные объекты Python. `load` читает файл, `loads` — строку. `students` стал списком словарей, поэтому применимы индексы и ключи Python.

</details>

## 2. Синтаксис JSON и проверка структуры

У JSON есть два независимых уровня ошибок.

1. **Синтаксис:** пропущена скобка, использованы одинарные кавычки, поставлена лишняя запятая или в документе записано несколько верхнеуровневых значений. Парсер остановится с `json.JSONDecodeError`, где указаны строка и столбец.
2. **Структура данных:** JSON синтаксически корректен, но вместо ожидаемого списка пришёл объект, отсутствует `metrics` или поле `loss` имеет неверный тип. Парсер такой документ прочитает, поэтому структуру проверяет программа.

После `json.load` нет специального «JSON-объекта»: остаются обычные словари и списки Python. В `run["metrics"]["loss"]` сначала берётся словарь `metrics`, затем его поле `loss`. Если хотя бы одного обязательного ключа нет, программа получает `KeyError` и останавливается. На реальных данных одна неполная запись не должна молча ломать обработку всего файла.

Для обязательного поля используют квадратные скобки и явно обрабатывают ошибочную запись. Для необязательного поля используют `record.get("score")`: при отсутствии получится `None`.

Правильный JSON всё равно может иметь неожиданную структуру. До вычислений проверяют верхний тип, обязательные поля и типы значений. Для числа здесь используется `type(value) in (int, float)`: простой `isinstance(value, (int, float))` также принял бы `True`, потому что `bool` в Python является подтипом `int`.

In [ ]:
text = """
[
  {"name": "alpha", "metrics": {"loss": 0.31, "accuracy": 0.91}},
  {"name": "beta", "metrics": {"loss": 0.27}},
  {"name": "gamma", "metrics": {"accuracy": 0.88}}
]
"""

runs = json.loads(text)
if not isinstance(runs, list):
    raise TypeError("top-level value must be a list")

valid_runs = []
errors = []
for index, run in enumerate(runs):
    metrics = run.get("metrics") if isinstance(run, dict) else None
    loss = metrics.get("loss") if isinstance(metrics, dict) else None
    if type(loss) not in (int, float):
        errors.append({"index": index, "reason": "missing or invalid loss"})
        continue
    valid_runs.append({"name": run.get("name"), "loss": loss})

print("valid:", valid_runs)
print("errors:", errors)

#### ❓ **Вопрос**

Что произойдёт при `json.loads(text)`? Парсер сообщит обо всех проблемах сразу?

```python
text = """
[
  {"name": "alpha", "metrics": {"loss": 0.31, "accuracy": 0.91}},
], [  {"name": "beta", "metrics": {"loss": 0.27}}]
"""
```

<details><summary>Ответ</summary>

Возникнет `JSONDecodeError`. Сначала парсер остановится на запятой перед `]`. После её удаления останется вторая ошибка: два верхнеуровневых массива. Нужен один общий массив либо один массив из двух вложенных массивов.

</details>

## 3. XML: чем он отличается от JSON

XML — текстовое дерево элементов. В `<server id="web-01"><cpu>8</cpu></server>` `server` и `cpu` — теги, `id` — атрибут, `8` — текст, а `cpu` — дочерний элемент.

| | JSON | XML |
|---|---|---|
| Модель | объекты, массивы и простые типы | дерево элементов, атрибутов и текста |
| Числа и `true` | имеют собственный тип | после чтения текст обычно остаётся строкой |
| Объём | обычно компактнее | обычно многословнее |
| Сильная сторона | API и данные для приложений | документы, namespaces, смешанный текст и строгие отраслевые схемы |
| Где встречается | REST API, настройки, обмен между сервисами | SOAP, офисные форматы, SVG, Maven, государственные и корпоративные системы |

JSON удобнее для обычных структур приложения. XML полезнее, когда важны порядок и разметка документа, атрибуты, пространства имён или уже существует отраслевой стандарт. Один формат не является «новой версией» другого.

Чёткую структуру можно задать и для JSON через JSON Schema. В XML часто используют XML Schema (XSD): она описывает допустимые элементы, порядок, обязательные атрибуты и типы. Подробнее — в «Дополнительно».

`ET.parse(path)` читает XML-файл, `ET.fromstring(text)` — строку, `getroot()` возвращает корень. Без схемы `ElementTree` возвращает текст как `str`, поэтому `findtext("price")` нужно явно преобразовать в число.

Стандартная библиотека поддерживает полезную часть XPath:

- `find(path)` возвращает первый элемент или `None`;
- `findall(path)` возвращает список;
- `.//book` ищет `book` на любой глубине;
- `./catalog[@kind='current']/book` выбирает ветку по атрибуту;
- `.//book[@id='b2']` ищет элемент по `id`;
- `element.get("id")` читает атрибут, а `get("lang", "unknown")` задаёт значение при его отсутствии.

При создании XML используют `ET.Element`, `ET.SubElement` и `ElementTree.write`. Полный XPath через `lxml` вынесен в «Дополнительно».

In [ ]:
import xml.etree.ElementTree as ET

xml = """
<library>
  <catalog kind="current">
    <book id="b1" lang="ru"><title>Python</title><price>1200</price></book>
    <book id="b2"><title>Linux</title><price>900</price></book>
  </catalog>
  <catalog kind="archive">
    <book id="b3" lang="en"><title>Old Unix</title><price>1500</price></book>
  </catalog>
</library>
"""

root = ET.fromstring(xml)
for book in root.findall("./catalog[@kind='current']/book"):
    print(
        book.get("id"),
        book.get("lang", "unknown"),
        book.findtext("title"),
        int(book.findtext("price")),
    )

book_b2 = root.find(".//book[@id='b2']")
print("found:", book_b2.findtext("title"))
print("missing:", root.find(".//book[@id='missing']"))

#### ❓ **Вопрос**

Чем отличаются `find(".//book[@id='b2']")` и `findall("./catalog[@kind='current']/book")`? Что вернёт `find` без совпадения?

<details><summary>Ответ</summary>

Первое выражение возвращает первую книгу с заданным `id` на любой глубине. Второе возвращает список книг только из текущего каталога. Без совпадения `find` вернёт `None`.

</details>

## 4. HTTP, REST API, методы и query-параметры

API — договор между клиентом и сервером: какой адрес вызвать, какой метод и параметры передать, какие статусы и структуру ответа ожидать. REST — распространённый стиль такого договора, где URL обычно обозначает ресурс, например `/users/42`, а HTTP-метод — действие над ним.

**Безопасный** метод предназначен только для чтения. **Идемпотентный** метод можно повторить несколько раз: итоговое состояние сервера будет тем же, хотя журнал или ответ могут отличаться.

| Метод | Обычное назначение | Безопасен | Идемпотентен | Тело запроса | Кэш ответа |
|---|---|---:|---:|---|---|
| `GET` | получить ресурс | да | да | нет | да |
| `HEAD` | получить только заголовки как у `GET` | да | да | нет | да |
| `POST` | создать, изменить или запустить действие | нет | нет | обычно да | нет |
| `PUT` | полностью создать или заменить ресурс по URL | нет | да | обычно да | нет |
| `PATCH` | частично изменить ресурс | нет | нет | обычно да | нет |
| `DELETE` | удалить ресурс | нет | да | обычно нет | нет |
| `OPTIONS` | узнать возможности взаимодействия | да | да | обычно нет | нет |
| `QUERY` | выполнить серверный запрос с описанием в теле | да | да | ожидается | да |
| `TRACE` | диагностическая петля | да | да | запрещено | нет |
| `CONNECT` | открыть сетевой туннель | нет | нет | специальная семантика | нет |

Практическое правило курса: **`POST` меняет состояние сервера и автоматически не повторяется**. Если автор конкретного API сделал читающий `POST`, это его частная договорённость, которую нельзя определить по самому методу.

`QUERY` — идемпотентная замена `POST` для чтения или вычисления, когда описание запроса слишком велико или неудобно для URL и должно находиться в теле. Это не замена созданию и изменению данных. Метод новый, поэтому сервер может его ещё не поддерживать.

**URI query-параметры** — другое понятие: это часть URL после `?`, например `/items?category=book&page=2`. В `requests` их передают словарём `params`, библиотека сама кодирует значения. `json` кодирует JSON-тело запроса.

Query-параметры видны в URL и журналах, поэтому секреты туда не помещают. Авторизацию разберём вместе со статусами ответа в следующем блоке.

In [ ]:
import requests

response = requests.get(
    "https://httpbin.org/get",
    params={"course": "python", "page": 2},
    timeout=10,
)
response.raise_for_status()

data = response.json()
print(response.status_code)
print(response.url)
print(data["args"])

# Только для сервера с поддержкой QUERY:
# requests.request("QUERY", url, json={"price_lt": 1000}, timeout=10)

#### ❓ **Вопрос**

Соединение оборвалось до ответа. Какие из `GET`, `POST`, `QUERY` можно автоматически повторить?

<details><summary>Ответ</summary>

`GET` и `QUERY`. `POST` считаем изменившим сервер: повтор может создать второй объект или повторно запустить действие.

</details>

## 5. Ошибки транспорта, HTTP и данных

Слово «ошибка» скрывает разные этапы:

1. **Транспорт:** нет HTTP-ответа — `ConnectionError`, `Timeout`.
2. **HTTP:** сервер ответил статусом 4xx или 5xx — `raise_for_status()` создаёт `HTTPError`.
3. **Формат тела:** ответ получен, но тело не JSON — `JSONDecodeError`.
4. **Валидация данных:** JSON корректен, но структура или типы неверны.

Первая цифра статуса задаёт класс. **2xx — успешные ответы, а не ошибки.** 4xx означает проблему запроса клиента, 5xx — проблему на стороне сервера или промежуточного сервера.

| Статус | Название | Что обычно означает |
|---:|---|---|
| `200` | OK | запрос выполнен, результат обычно находится в теле |
| `201` | Created | ресурс создан |
| `204` | No Content | запрос выполнен, тела ответа нет |
| `400` | Bad Request | неверный синтаксис или параметры запроса |
| `401` | Unauthorized | нет токена или токен неверный |
| `403` | Forbidden | сервер узнал клиента, но запрещает действие |
| `404` | Not Found | ресурс по этому URL не найден |
| `422` | Unprocessable Content | формат понятен, но данные не прошли валидацию |
| `429` | Too Many Requests | превышен лимит запросов |
| `500` | Internal Server Error | внутренняя ошибка сервера |
| `502` | Bad Gateway | промежуточный сервер получил плохой ответ |
| `503` | Service Unavailable | сервис временно недоступен |

Токен авторизации передают в **заголовке**, а не в URL:

```python
token = os.environ["API_TOKEN"]
headers = {"Authorization": f"Bearer {token}"}
response = requests.get(url, headers=headers, timeout=10)
```

Здесь достаточно знать один заголовок `Authorization`. Остальные частые заголовки вынесены в «Дополнительно».

Даже статус `200` не гарантирует JSON или нужные поля. Порядок проверки: получить ответ → проверить статус → разобрать формат → проверить структуру.

In [ ]:
def load_items(url, token):
    headers = {"Authorization": f"Bearer {token}"}

    try:
        response = requests.get(
            url,
            headers=headers,
            timeout=10,
        )
        response.raise_for_status()
    except requests.Timeout:
        return "transport: timeout"
    except requests.ConnectionError:
        return "transport: connection"
    except requests.HTTPError as error:
        return f"http: {error.response.status_code}"

    try:
        data = response.json()
    except requests.exceptions.JSONDecodeError:
        return "format: not JSON"

    if not isinstance(data, dict):
        return "validation: top level is not an object"
    if not isinstance(data.get("items"), list):
        return "validation: items is not a list"
    return data["items"]

#### ❓ **Вопрос**

Чем отличаются ответы `401` и `403`? Почему после `200` всё равно нужно отдельно разбирать JSON и проверять его структуру?

<details><summary>Ответ</summary>

`401` означает, что сервер не получил подходящие данные авторизации: например, токена нет или он неверный. `403` означает, что сервер распознал клиента, но у него нет права на действие. `200` подтверждает успех на уровне HTTP, но тело может оказаться не JSON или иметь неожиданную структуру.

</details>

## 6. HTML и Beautiful Soup

Корректный XHTML можно прочитать как XML. Обычный веб-HTML живёт по другим правилам: некоторые закрывающие теги необязательны, `<br>` не требует `</br>`, а HTML-парсер восстанавливает несовершенную разметку.

Beautiful Soup строит такое дерево и помогает получить видимый текст. `decompose()` удаляет `script` и `style`, `get_text(" ", strip=True)` собирает оставшийся текст.

In [ ]:
from bs4 import BeautifulSoup

html = """
<html><head>
  <style>.hidden { display: none; }</style>
  <script>console.log("not visible")</script>
</head><body>
  <h1>Новости</h1>
  <p>Первая заметка<br>
  <p>Вторая заметка
</body></html>
"""

try:
    ET.fromstring(html)
except ET.ParseError as error:
    print("XML parser:", error)

soup = BeautifulSoup(html, "html.parser")
for node in soup.select("script, style"):
    node.decompose()
print("HTML parser:", soup.get_text(" ", strip=True))

#### ❓ **Вопрос**

В строке `html` из ячейки выше теги `<br>` и `<p>` не закрыты по правилам XML. Почему `ET.fromstring(html)` завершается с `ParseError`, а `BeautifulSoup(html, "html.parser")` всё равно строит дерево? Какое содержимое удалят вызовы `decompose()` для `script` и `style`?

<details><summary>Ответ</summary>

XML требует правильно закрывать каждый элемент, поэтому `ElementTree` останавливается на неверной структуре. HTML-парсер знает правила HTML и восстанавливает дерево. `decompose()` полностью удалит теги `script` и `style` вместе с JavaScript-кодом и CSS внутри них.

</details>

## 7. OpenAI API и OpenAI-compatible серверы

Библиотека `openai` умеет подключаться не только к серверам OpenAI, но и к OpenAI-compatible серверам, которые реализуют такой же HTTP API. Для подключения нужны три значения:

- `OPENAI_BASE_URL` — полный базовый адрес API, который выдал сервер;
- `OPENAI_API_KEY` — токен авторизации;
- `OPENAI_MODEL` — имя модели, доступной на выбранном сервере.

**Повторяем! Все секретные ключи в код записывать нельзя.** Адрес, токен и модель сохраняют в переменных окружения:

```bash
export OPENAI_BASE_URL='https://api.openai.com/v1/'
export OPENAI_API_KEY='ваш-ключ'
export OPENAI_MODEL='gpt-5.6'
```

Для другого совместимого сервера адрес и модель берут из его документации. Например, это может быть `OPENAI_BASE_URL='http://localhost:8000/v1/'`. `/v1` нельзя бездумно добавлять или удалять: это часть адреса, заданного конкретным сервером.

В примере используется Chat Completions API: SDK отправляет `POST` с JSON-полями `model` и `messages`, а текст ответа находится в `completion.choices[0].message.content`.

In [ ]:
import os
from openai import OpenAI

base_url = os.environ["OPENAI_BASE_URL"].rstrip("/")
api_key = os.environ["OPENAI_API_KEY"]
model = os.environ["OPENAI_MODEL"]

messages = [
    {
        "role": "user",
        "content": "Коротко поприветствуй студентов курса Python.",
    }
]

client = OpenAI(
    base_url=base_url,
    api_key=api_key,
)

completion = client.chat.completions.create(
    model=model,
    messages=messages,
)
print(completion.choices[0].message.content)

# Тот же запрос без SDK, через обычный requests:
# import requests
# response = requests.post(
#     f"{base_url}/chat/completions",
#     headers={"Authorization": f"Bearer {api_key}"},
#     json={"model": model, "messages": messages},
#     timeout=30,
# )
# response.raise_for_status()
# completion_json = response.json()
# print(completion_json["choices"][0]["message"]["content"])

#### ❓ **Вопрос**

Какие три переменные окружения определяют, к какому серверу и модели подключится программа? Где в коде находится запрос пользователя, а где извлекается текст ответа?

<details><summary>Ответ</summary>

`OPENAI_BASE_URL` задаёт адрес сервера, `OPENAI_API_KEY` — токен, `OPENAI_MODEL` — модель. Текст запроса находится в `messages[0]["content"]`, ответ — в `completion.choices[0].message.content`.

</details>

## Дополнительно

### XML Schema (XSD)

JSON Schema задаёт структуру JSON, XSD — структуру XML. XSD описывает порядок элементов, обязательные атрибуты и типы:

```xml
<xs:element name="price" type="xs:decimal"/>
<xs:attribute name="id" type="xs:string" use="required"/>
```

`ElementTree` не валидирует XSD. Для этого используют `lxml` или `xmlschema`. После чтения значения всё равно явно преобразуют в типы Python.

### Полный XPath через lxml

`ElementTree` покрывает простые пути. Сравнения, функции, текст и атрибуты доступны в `lxml`:

```bash
uv add lxml
```

```python
from lxml import etree

document = etree.parse("books.xml")
titles = document.xpath(
    "/library/catalog[@kind='current']/book[price > 1000]/title/text()"
)
book_ids = document.xpath("//book/@id")
```

### HTTP-заголовки и токены

`Accept` задаёт ожидаемый формат, `Content-Type` — формат тела, `Authorization` — авторизацию, `User-Agent` — имя клиента.

```python
headers = {
    "Authorization": f"Bearer {os.environ['API_TOKEN']}",
    "Accept": "application/json",
}
response = requests.get(url, headers=headers, timeout=10)
```

Токен не записывают в код, query-параметры или вывод программы.

### CSS-селекторы Beautiful Soup и XPath

`select(".product")` находит все карточки, `select_one(".price")` — первый элемент или `None`, `select("tbody tr")` — строки таблицы. `get("href")` читает ссылку, `get_text(" ", strip=True)` — текст. Относительный адрес превращают в абсолютный через `urljoin(base_url, href)`.

Beautiful Soup не поддерживает XPath. Даже в `BeautifulSoup(html, "lxml")` слово `lxml` выбирает парсер, но результат всё равно остаётся деревом Beautiful Soup с методами `select()` и `select_one()`. Если нужен XPath по HTML, страницу читают через `lxml.html`:

```python
from lxml import html as lxml_html

tree = lxml_html.fromstring(html)
titles = tree.xpath("//article[@class='product']//h2/text()")
```

Обычно одну страницу не разбирают сначала Beautiful Soup, а затем повторно `lxml`: под задачу выбирают CSS-селекторы Beautiful Soup или XPath `lxml`.

### YAML

YAML часто используют в конфигурациях, CI, Docker Compose и Kubernetes. `.yml` и `.yaml` равнозначны. Формат поддерживает комментарии и обычно короче JSON, но зависит от правильных отступов.

```bash
uv add pyyaml
```

```python
import yaml

with open("config.yml", encoding="utf-8") as file:
    config = yaml.safe_load(file)
```

Для чужих данных используют `safe_load`, затем проверяют структуру и типы как у JSON.

### Пагинация, кэш и связанные страницы

Конец страниц API обозначается пустым списком, полем `next` или числом страниц — правило берут из документации. Дубли удаляют словарём по `id`.

Проверенный ответ сначала пишут во временный файл, затем вызывают `os.replace("cache.json.tmp", "cache.json")`. Старый кэш не портится при ранней ошибке; `Path` здесь не нужен.

В H5 `Path` используется только для локальных HTML-файлов: `resolve()` получает абсолютный путь, `parent / href` строит соседний, `as_uri()` превращает путь в `file://` URL.

### OpenAI API — модели, запрос и ответ

#### Как узнать доступные модели

Клиент запрашивает endpoint `GET /v1/models`:

```python
models = client.models.list()

for item in models.data:
    print(item.id)
```

Полученный `id` записывают в `OPENAI_MODEL`. Не каждый compatible-сервер реализует список моделей; если сервер отвечает `404`, имя модели берут из его документации или настроек.

#### Как сформировать запрос

Chat Completions принимает имя модели и список сообщений. `developer` задаёт правило работы, `user` — запрос пользователя:

```python
messages = [
    {"role": "developer", "content": "Отвечай одним предложением."},
    {"role": "user", "content": "Что такое JSON?"},
]

completion = client.chat.completions.create(
    model=os.environ["OPENAI_MODEL"],
    messages=messages,
)
```

SDK превращает словари Python в JSON и отправляет `POST /v1/chat/completions`.

#### Как получить ответ

Сервер возвращает JSON-объект. SDK превращает его в объект Python:

```python
answer = completion.choices[0].message.content
print(answer)

print(completion.model_dump_json(indent=2))
```

В полном ответе полезны `choices` с результатами, `model` с фактически использованной моделью, `usage` с количеством токенов и `finish_reason` с причиной завершения.

OpenAI-compatible означает совместимый формат API, но не гарантирует поддержку всех endpoints и параметров. Chat Completions обычно поддерживается шире; Responses API используют, если он явно заявлен выбранным сервером.

- [Список моделей OpenAI](https://developers.openai.com/api/docs/models)
- [Chat Completions API](https://developers.openai.com/api/reference/resources/chat/subresources/completions/methods/create)
- [Официальный quickstart OpenAI](https://developers.openai.com/api/docs/quickstart)